# Studio 2 — Kompetisi Klasifikasi Sampah: Percobaan ConvNeXt-Tiny (crop 224)

Deep Learning (IF25-40401) · Teknik Informatika ITERA · Semester Ganjil 2026
**Kelompok:** 11 (YOLO)

Notebook ini menjalankan **satu percobaan** di Google Colab, mulai dari runtime kosong sampai bobot
siap di-commit. Urutannya: pemeriksaan runtime, pengambilan repositori, penyiapan dataset,
pemeriksaan split validasi beku, pelatihan, evaluasi resmi, penyusunan grafik, pemeriksaan ukuran
berkas, lalu promosi ke `submission/`.

Pengaturan di notebook ini sudah tetap dan tidak perlu disunting karena percobaan ini pernah
dijalankan; angka hasilnya tercatat pada tabel di bagian akhir. Notebook pasangannya,
`percobaan_effnetv2s.ipynb` (EfficientNetV2-S (crop 288)), memiliki alur yang sama dengan isi sel konfigurasi yang berbeda.

## Prasyarat

1. GPU aktif: menu `Runtime` → `Change runtime type` → `T4 GPU`.
2. Berkas `train.zip` tersimpan pada Google Drive di `MyDrive/Studio 2/Dataset/train.zip`.
   Berkas ini diunduh sekali dari tautan resmi halaman kompetisi; dataset tidak disalin ke repositori.
3. Folder arsip bobot pada Drive: `MyDrive/Studio 2/Ckpt/`.

## Ketentuan kompetisi yang dijaga notebook ini

| Ketentuan | Penerapan di notebook ini |
|---|---|
| Dataset tidak diunggah ulang ke repositori publik | Dataset dibaca dari Google Drive; hanya kode dan bobot yang di-commit |
| `model.py` dan `model.pth` wajib sepasang | Promosi menyalin kedua berkas sekaligus; `evaluate.py` memuatnya dengan `strict=True` |
| Urutan kelas alfabetis | `0 botol_plastik · 1 gelas_plastik · 2 kertas_kardus · 3 saset_kemasan · 4 styrofoam` |
| Berkas maksimal 100 MB untuk GitHub | Langkah 9 menyimpan ulang bobot ke fp16 bila melewati batas, lalu menguji ulang skornya |
| Nilai ditentukan putaran Final (25 Oktober 2026) | Putaran Preview hanya menghasilkan umpan balik dan tidak dinilai |

## Catatan percobaan

| Butir | Isi |
|---|---|
| Arsitektur | ConvNeXt-Tiny (torchvision, pretrained ImageNet-1K) |
| Resolusi masukan | crop 224 px, resize 256 px |
| Hyperparameter | 20 epoch, batch 16, lr 0.0001, wd 0.01, augmentasi strong, patience 5, seed 42 |
| Hasil tercatat | macro-F1 0,9658 pada 413 gambar beku (epoch terbaik 5 dari 9 epoch tercatat) |
| Waktu pelatihan | ±115 detik per epoch pada GPU 4 GB (lokal) |
| Ukuran bobot fp32 | 106,2 MB |
| Tag berkas hasil | `convnext_run2` (hasil sebelumnya tersimpan pada tag `exp4_convnext`) |
| Status | Percobaan ini adalah berkas submission yang berlaku |


## 0. Alur notebook

| Langkah | Isi |
|---|---|
| 1 | Konfigurasi percobaan (sudah tetap) |
| 2 | Pemeriksaan runtime: GPU, VRAM, ruang disk |
| 3 | Mount Google Drive dan pengambilan repositori |
| 4a | Menemukan berkas dataset (termasuk melalui pintasan Drive) |
| 4b | Cadangan: unggah dataset per sesi |
| 4 | Penyiapan dataset (ekstraksi dan penghitungan ulang jumlah gambar) |
| 5 | Pemeriksaan split validasi beku (413 gambar, seed 42) |
| 6 | Pelatihan dengan `train.py` |
| 7 | Evaluasi resmi dengan `evaluate.py` |
| 8 | Grafik dan ringkasan per epoch |
| 9 | Pemeriksaan ukuran bobot dan konversi fp16 bila diperlukan |
| 10 | Promosi ke `submission/` (hanya bila unggul melewati ambang keyakinan) |
| 11 | Pengunduhan berkas dan perintah commit |
| 12 | Penanganan kendala yang umum terjadi |
| 13 | Tabel hasil antar-percobaan |


## 1. Konfigurasi

Sel ini sudah berisi setelan percobaan dan tidak perlu disunting. Setiap notebook percobaan
menyimpan konfigurasinya sendiri dan menunjuk berkas arsitekturnya secara eksplisit, sehingga tidak
ada kemungkinan notebook satu melatih arsitektur milik notebook lain.

**Mengubah resolusi atau arsitektur dilakukan pada berkas di folder `experiments/`, bukan di
notebook ini.** Angka `CROP` dan `RESIZE` hanya boleh berada di sana agar resolusi saat melatih
selalu sama dengan `get_transform()` yang dipakai penilai.


In [ ]:
import hashlib
import json
import pathlib
import shutil
import subprocess
import sys
import time

# ── Konfigurasi percobaan (sudah tetap) ───────────────────────────────────────
EXP_NAME     = 'convnext_run2'
CANDIDATE    = 'experiments/convnext_tiny.py'      # berkas berisi get_model() dan get_transform()
EPOCHS       = 20
BATCH_SIZE   = 16
LR           = 0.0001
WD           = 0.01
AUG          = 'strong'
PATIENCE     = 5
WORKERS      = 2                  # Colab menyediakan lebih dari satu inti CPU

# Skor tercatat kedua percobaan pada split validasi beku yang sama (413 gambar).
# Dipakai untuk memilih otomatis yang terbaik di antara keduanya.
SKOR_TERCATAT = {"convnext": 0.9658, "effnetv2s": 0.9595}
PERCOBAAN_INI = "convnext"      # percobaan yang dijalankan notebook ini

# ── Repositori ────────────────────────────────────────────────────────────────
REPO_URL = "https://github.com/SigitKurnia05/DL-RA_studio2-klasifikasi-sampah.git"
BRANCH   = "main"
REPO_DIR = pathlib.Path("/content") / REPO_URL.rsplit("/", 1)[-1].replace(".git", "")

# ── Lokasi pada Google Drive ──────────────────────────────────────────────────
DRIVE_DIR   = "/content/drive/MyDrive/Studio 2"
# Berkas dataset dicari otomatis pada sel 4a, termasuk melalui pintasan Drive.
# Tambahkan jalur manual pada daftar di bawah bila ingin menunjuknya langsung.
KANDIDAT_DATASET = []
CKPT_DIR    = pathlib.Path(f"{DRIVE_DIR}/Ckpt")
DATA_DIR    = pathlib.Path("/content/train")

# ── Berkas hasil: ditulis ke disk lokal Colab lebih dahulu (lebih cepat dari Drive) ──
WORK_DIR    = pathlib.Path(f"/content/studio2_{EXP_NAME}")
OUT_WEIGHTS = WORK_DIR / "model.pth"
HIST_OUT    = WORK_DIR / "history.json"
PLOT_OUT    = WORK_DIR / f"{EXP_NAME}_curves.png"
EVAL_OUT    = WORK_DIR / f"{EXP_NAME}_eval.json"
EVAL_FP16   = WORK_DIR / f"{EXP_NAME}_eval_fp16.json"
WORK_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)


def run(cmd, cwd=None):
    """Jalankan perintah dan tampilkan keluarannya secara mengalir.

    Args:
        cmd: daftar argumen perintah.
        cwd: direktori kerja; nilai bawaannya REPO_DIR.
    """
    print("$", " ".join(str(c) for c in cmd))
    print("-" * 72)
    p = subprocess.Popen([str(c) for c in cmd], cwd=str(cwd or REPO_DIR),
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("-" * 72)
    if p.returncode != 0:
        raise RuntimeError("perintah gagal (exit %d): %s"
                           % (p.returncode, " ".join(str(c) for c in cmd)))


print("percobaan        :", EXP_NAME)
print("berkas arsitektur:", CANDIDATE)
print("hyperparameter   : %d epoch | batch %d | lr %s | wd %s | augmentasi %s | patience %d"
      % (EPOCHS, BATCH_SIZE, LR, WD, AUG, PATIENCE))
print("percobaan ini    : %s (skor tercatat %.4f)" % (PERCOBAAN_INI, SKOR_TERCATAT[PERCOBAAN_INI]))
print("skor pembanding  : %.4f (%s)" % (max(SKOR_TERCATAT.values()),
        max(SKOR_TERCATAT, key=SKOR_TERCATAT.get)))
print("jalur dataset    :", KANDIDAT_DATASET or "(kosong; dicari otomatis pada sel 4a)")
print("berkas kerja     :", WORK_DIR)
print("arsip Drive      :", CKPT_DIR)


## 2. Pemeriksaan runtime

Bila `CUDA` bernilai `False`, pelatihan sebaiknya tidak dilanjutkan pada CPU: pelatihan
ConvNeXt-Tiny di CPU Colab dapat melebihi satu jam per epoch, sedangkan pada GPU T4 berkisar
dua hingga tiga menit per epoch.


In [ ]:
import sys
import torch

print("Python  :", sys.version.split()[0])
print("PyTorch :", torch.__version__)
print("CUDA    :", torch.cuda.is_available())

if torch.cuda.is_available():
    g = torch.cuda.get_device_properties(0)
    print("GPU     :", g.name)
    print("VRAM    : %.1f GB" % (g.total_memory / 1024 ** 3))
else:
    print()
    print("[PERINGATAN] GPU belum aktif.")
    print("  Runtime -> Change runtime type -> T4 GPU, lalu jalankan ulang sel ini.")

tot, used, free = shutil.disk_usage("/content")
print("\nRuang /content: %.1f GB bebas dari %.1f GB" % (free / 2 ** 30, tot / 2 ** 30))


## 3. Mount Google Drive dan pengambilan repositori

Bila repositori sudah pernah di-clone pada sesi yang sama, sel ini menarik perubahan terbaru
(`git pull`). Hash commit terakhir dicetak agar versi kode yang dipakai saat melatih dapat
ditelusuri pada riwayat commit.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

if REPO_DIR.exists():
    print(REPO_DIR, "sudah ada; menarik perubahan terbaru.")
    run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], cwd="/content")
else:
    run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], cwd="/content")

run(["git", "-C", REPO_DIR, "log", "-1", "--format=commit %h | %ad | %s", "--date=short"])


## 4a. Menemukan berkas dataset

Berkas `train.zip` dicari secara otomatis supaya notebook ini tidak bergantung pada satu nama folder
tertentu, dan agar **pintasan Drive** tetap terpakai. Bila dataset berada pada Drive anggota lain,
minta pemiliknya membagikan folder tersebut, lalu di Google Drive klik kanan folder itu →
`Organize` → `Add shortcut` → `My Drive`. Pintasan hasilnya berada di dalam My Drive Anda sehingga
ikut tertelusuri oleh sel di bawah.

Jalur manual juga bisa ditambahkan pada variabel `KANDIDAT_DATASET` di sel konfigurasi.


In [ ]:
# ── 4a. Menemukan berkas dataset ─────────────────────────────────────────────
# Ditelusuri: jalur manual pada KANDIDAT_DATASET, lalu MyDrive dan Shared drives
# sampai tiga tingkat. Pintasan Drive berada di dalam MyDrive sehingga ikut tercakup.
DRIVE = pathlib.Path("/content/drive")


def cari_train_zip(basis, kedalaman=3, batas=4000):
    """Telusuri `basis` sampai `kedalaman` tingkat; kumpulkan berkas bernama train.zip.

    Args:
        basis: folder awal penelusuran.
        kedalaman: kedalaman maksimum penelusuran.
        batas: jumlah berkas maksimum yang dikumpulkan (mencegah penelusuran terlalu lama).

    Returns:
        list[pathlib.Path]: berkas train.zip yang ditemukan.
    """
    hasil = []

    def turun(folder, sisa):
        if len(hasil) >= batas:
            return
        try:
            for p in folder.iterdir():
                if len(hasil) >= batas:
                    return
                try:
                    if p.is_file() and p.name == "train.zip":
                        hasil.append(p)
                    elif p.is_dir() and sisa > 0:
                        turun(p, sisa - 1)
                except OSError:
                    continue
        except (PermissionError, OSError):
            return

    turun(pathlib.Path(basis), kedalaman)
    return hasil


KANDIDAT = [pathlib.Path(p) for p in KANDIDAT_DATASET if pathlib.Path(p).is_file()]
print("Jalur manual pada KANDIDAT_DATASET:", len(KANDIDAT), "berkas")

for basis in ("MyDrive", "Shareddrives"):
    akar = DRIVE / basis
    if not akar.exists():
        continue
    ditemukan = cari_train_zip(akar)
    print("%-58s %d berkas train.zip" % (str(akar), len(ditemukan)))
    KANDIDAT += [p for p in ditemukan if p not in KANDIDAT]

DATASET_ZIP = KANDIDAT[0] if KANDIDAT else None

if DATASET_ZIP is None:
    print()
    print("[BELUM DITEMUKAN] train.zip tidak ada pada Drive Anda. Pilihan yang tersedia:")
    print("  1. Minta pemilik dataset membagikan folder 'Studio 2', lalu di Google Drive klik")
    print("     kanan folder tersebut -> Organize -> Add shortcut -> My Drive, dan ulangi sel ini.")
    print("  2. Salin train.zip (311 MB) ke Drive Anda sendiri, misalnya")
    print("     MyDrive/Studio 2/Dataset/train.zip.")
    print("  3. Unggah per sesi: jalankan sel 4b di bawah, lalu ulangi sel ini.")
    print()
    print("Isi tingkat pertama MyDrive Anda (untuk memeriksa apakah pintasan terbaca):")
    try:
        for p in sorted((DRIVE / "MyDrive").iterdir())[:25]:
            print("  [%s] %s" % ("folder" if p.is_dir() else "berkas", p.name))
    except (PermissionError, OSError) as galat:
        print("  tidak dapat dibaca:", galat)
    print()
    print("Bila folder 'Studio 2' tidak muncul di daftar itu, pintasannya belum ada di My Drive.")
else:
    print()
    print("berkas dataset :", DATASET_ZIP)
    print("ukuran         : %.1f MB" % (DATASET_ZIP.stat().st_size / 2 ** 20))
    if "Shareddrives" in str(DATASET_ZIP) or "shortcut" in str(DATASET_ZIP).lower():
        print("catatan        : jalur ini berasal dari pintasan atau Drive bersama;")
        print("                 tidak perlu menyalin berkas apa pun.")


## 4b. Cadangan: unggah dataset per sesi

Dijalankan hanya bila sel 4a gagal menemukan berkasnya. Berkas yang diunggah tidak permanen:
hilang ketika sesi Colab berakhir, sehingga harus diulang pada sesi berikutnya.


In [ ]:
# ── 4b. Cadangan: unggah dataset per sesi ────────────────────────────────────
# Dijalankan hanya bila sel 4a belum menemukan berkasnya.
if DATASET_ZIP is None:
    from google.colab import files as _files

    print("Pilih berkas data/train.zip dari laptop Anda (sekitar 311 MB).")
    unggah = _files.upload()
    if unggah:
        nama = next(iter(unggah))
        DATASET_ZIP = pathlib.Path("/content") / nama
        print("berkas dataset:", DATASET_ZIP)
        print("ukuran        : %.1f MB" % (DATASET_ZIP.stat().st_size / 2 ** 20))
    else:
        print("Tidak ada berkas yang diunggah.")
else:
    print("Dataset sudah ditemukan pada", DATASET_ZIP, "; sel ini tidak perlu dijalankan.")


## 4. Penyiapan dataset

Dataset dibaca dari berkas zip pada Google Drive lalu diekstrak ke disk Colab (`/content/train`),
karena pembacaan ribuan gambar langsung dari Drive jauh lebih lambat. Setelah ekstraksi, jumlah
gambar tiap kelas dihitung ulang dan wajib sama dengan snapshot saat split beku dibuat
(2.753 gambar). Bila tidak sama, notebook berhenti agar hasilnya tidak dibandingkan secara keliru
dengan percobaan sebelumnya.


In [ ]:
import zipfile

CLASSES = ["botol_plastik", "gelas_plastik", "kertas_kardus", "saset_kemasan", "styrofoam"]
EXPECT  = {"botol_plastik": 523, "gelas_plastik": 521, "kertas_kardus": 365,
           "saset_kemasan": 673, "styrofoam": 671}

if not DATA_DIR.exists() or not all((DATA_DIR / c).is_dir() for c in CLASSES):
    if DATASET_ZIP is None or not pathlib.Path(DATASET_ZIP).is_file():
        raise FileNotFoundError(
            "[GAGAL] Berkas dataset belum ditetapkan.\n"
            "        Jalankan sel 4a (pencarian otomatis) atau sel 4b "
            "(unggah per sesi) lebih dahulu.")
    print("Mengekstrak %s ke %s ..." % (DATASET_ZIP, DATA_DIR.parent))
    with zipfile.ZipFile(DATASET_ZIP) as z:
        z.extractall(DATA_DIR.parent)
    print("Ekstraksi selesai.")
else:
    print(DATA_DIR, "sudah ada; ekstraksi dilewati.")

jumlah = {c: len(list((DATA_DIR / c).glob("*.jpg"))) for c in CLASSES}
print("\nDataset: %s | total %d gambar\n" % (DATA_DIR, sum(jumlah.values())))
for c in CLASSES:
    tanda = "OK  " if jumlah[c] == EXPECT[c] else "BEDA"
    print("  [%s] %-16s %4d gambar" % (tanda, c, jumlah[c])
          + ("" if jumlah[c] == EXPECT[c] else "  (seharusnya %d)" % EXPECT[c]))

if sum(jumlah.values()) != sum(EXPECT.values()):
    raise RuntimeError("Jumlah gambar tidak sama dengan snapshot split beku; proses dihentikan.")
print("\nDataset sesuai dengan split beku (2.753 gambar).")


## 5. Pemeriksaan split validasi beku

Split validasi tidak pernah dibagi ulang. Daftarnya dibekukan pada `splits/val.txt`: 413 gambar,
seed 42, hash kanonik `36bee95ec932553bd3bf89d8eabe8c1d562d1343f65a442235c1df403be7a423`.
Sel ini menghitung ulang hash tersebut dan membandingkannya dengan berkas. Bila berbeda, split
atau dataset telah berubah dan seluruh angka percobaan sebelumnya tidak lagi sebanding.

Gambar validasi juga disalin ke `my_val/` agar `evaluate.py` dapat dijalankan. Folder tersebut
masuk `.gitignore` sehingga tidak ikut di-commit.


In [ ]:
run([sys.executable, REPO_DIR / "make_split.py", "--check", "--data", DATA_DIR])
run([sys.executable, REPO_DIR / "make_split.py", "--materialize", "my_val", "--data", DATA_DIR])

print("my_val/ siap:", len(list((REPO_DIR / "my_val").glob("*/*.jpg"))), "gambar")


## 6. Pelatihan

`train.py` membaca daftar validasi dari `splits/val.txt` (tidak pernah membentuk split sendiri),
menyimpan bobot dengan **macro-F1 terbaik** (bukan akurasi dan bukan epoch terakhir), dan berhenti
lebih awal bila macro-F1 validasi tidak meningkat selama `patience` epoch.

Augmentasi `strong` terdiri atas RandomResizedCrop 0,5–1,0, flip vertikal, RandAugment(2, 9), dan
RandomErasing p=0,25. Pada backbone berukuran belasan hingga puluhan juta parameter dengan 2.340
gambar latih, augmentasi tersebut bersama weight decay merupakan pengendali overfitting utama;
tanpa keduanya, kurva validasi melonjak seperti yang terjadi pada percobaan 1.


In [ ]:
# model.py disalin ke folder kerja agar evaluate.py dapat menunjuk folder ini langsung
shutil.copyfile(REPO_DIR / CANDIDATE, WORK_DIR / "model.py")

cmd = [
    sys.executable, REPO_DIR / "train.py",
    "--data",        DATA_DIR,
    "--split",       "splits/val.txt",
    "--model",       CANDIDATE,
    "--out",         OUT_WEIGHTS,
    "--epochs",      EPOCHS,
    "--bs",          BATCH_SIZE,
    "--lr",          LR,
    "--wd",          WD,
    "--aug",         AUG,
    "--patience",    PATIENCE,
    "--history-out", HIST_OUT,
    "--workers",     WORKERS,
]

t0 = time.time()
run(cmd)
print("Total waktu pelatihan: %.1f menit" % ((time.time() - t0) / 60))
print("Bobot terbaik        : %s (%.1f MB)" % (OUT_WEIGHTS, OUT_WEIGHTS.stat().st_size / 2 ** 20))


## 7. Evaluasi resmi

Skrip ini identik dengan yang dijalankan asisten dosen. Pada Colab maupun laptop, ia hanya menguji
split lokal `my_val/` (413 gambar), bukan validation set tersembunyi (±740 gambar). Fungsinya
memilih model, bukan memperkirakan skor akhir.

Macro-F1 adalah rata-rata F1 dari kelima kelas, sehingga kelas kecil seperti `kertas_kardus`
(365 gambar latih) berbobot sama dengan kelas besar.


In [ ]:
if not (WORK_DIR / "model.py").is_file() or not OUT_WEIGHTS.is_file():
    raise SystemExit("Bobot belum ada; jalankan sel pelatihan (langkah 6) lebih dahulu.")

run([sys.executable, REPO_DIR / "evaluate.py",
     "--submission", WORK_DIR,
     "--data", "my_val",
     "--out", EVAL_OUT])

m = json.loads(EVAL_OUT.read_text(encoding="utf-8"))
print("Skor lokal: macro-F1 %.4f | akurasi %.4f | %d gambar | %.1f detik"
      % (m["macro_f1"], m["accuracy"], m["n_images"], m["seconds"]))


## 8. Grafik dan ringkasan per epoch

Panel kiri memperlihatkan train loss dan val loss; jarak keduanya merupakan indikasi overfitting.
Panel tengah memperlihatkan macro-F1 dan akurasi, dengan garis putus-putus sebagai baseline asisten
(ResNet-18 ≈ 0,92 pada validation set tersembunyi). Pembanding tersebut tidak setara karena
himpunannya berbeda, sehingga hanya dipakai sebagai patokan. Panel kanan adalah confusion matrix;
dua kelas yang paling sering tertukar adalah `kertas_kardus` dan `styrofoam`, keduanya berwarna
terang dan bertekstur serupa.


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

hist = json.loads(HIST_OUT.read_text(encoding="utf-8"))
ep  = [h["epoch"] for h in hist]
tr  = [h["train_loss"] for h in hist]
va  = [h["val_loss"] for h in hist]
f1  = [h["val_macro_f1"] for h in hist]
acc = [h["val_acc"] for h in hist]
puncak = max(range(len(f1)), key=lambda i: f1[i])

print("percobaan        :", EXP_NAME)
print("epoch tercatat   :", len(ep))
print("macro-F1 terbaik : %.4f (epoch %d)" % (f1[puncak], ep[puncak]))
print("akurasi epoch itu: %.4f" % acc[puncak])
print("waktu per epoch  : %.1f detik (rata-rata)" % (sum(h["detik"] for h in hist) / len(hist)))
print()
print("%3s %10s %9s %9s %8s  %s" % ("Ep", "TrainLoss", "ValLoss", "MacroF1", "Acc", "Rekor"))
print("-" * 52)
for h in hist:
    print("%3d %10.4f %9.4f %9.4f %8.4f  %s" % (h["epoch"], h["train_loss"], h["val_loss"],
                                                h["val_macro_f1"], h["val_acc"],
                                                "BEST" if h["epoch"] == ep[puncak] else ""))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
ax[0].plot(ep, tr, "o-", label="train")
ax[0].plot(ep, va, "s-", label="val")
ax[0].set_title("Loss per epoch"); ax[0].set_xlabel("epoch"); ax[0].set_ylabel("cross-entropy")
ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].xaxis.set_major_locator(MaxNLocator(integer=True))

ax[1].plot(ep, f1, "o-", color="#2ca02c", label="macro-F1")
ax[1].plot(ep, acc, "^--", color="#7f7f7f", label="akurasi")
ax[1].axhline(0.92, color="#d62728", ls=":", lw=1.5, label="baseline asisten ~0,92")
ax[1].scatter([ep[puncak]], [f1[puncak]], s=180, marker="*", color="#ff7f0e", zorder=5,
              label="terbaik %.4f (ep %d)" % (f1[puncak], ep[puncak]))
ax[1].set_title("Macro-F1 dan akurasi"); ax[1].set_xlabel("epoch"); ax[1].set_ylabel("skor")
ax[1].legend(fontsize=9); ax[1].grid(alpha=0.3); ax[1].xaxis.set_major_locator(MaxNLocator(integer=True))

if EVAL_OUT.is_file():
    mm = json.loads(EVAL_OUT.read_text(encoding="utf-8"))
    cm = mm["confusion_matrix"]
    label = [c[:6] for c in CLASSES]
    ax[2].imshow(cm, cmap="Blues")
    batas = max(max(r) for r in cm) / 2
    for i in range(len(cm)):
        for j in range(len(cm)):
            ax[2].text(j, i, cm[i][j], ha="center", va="center",
                       color="white" if cm[i][j] > batas else "black")
    ax[2].set_xticks(range(len(label))); ax[2].set_xticklabels(label, rotation=30, ha="right")
    ax[2].set_yticks(range(len(label))); ax[2].set_yticklabels(label)
    ax[2].set_title("Confusion matrix (macro-F1 %.4f)" % mm["macro_f1"])
    ax[2].set_xlabel("prediksi"); ax[2].set_ylabel("label asli")
else:
    ax[2].text(0.5, 0.5, "berkas evaluasi belum tersedia", ha="center", va="center")
    ax[2].set_title("Confusion matrix")

fig.suptitle("%s - lr %s, wd %s, augmentasi %s, %s" % (EXP_NAME, LR, WD, AUG, CANDIDATE))
fig.tight_layout()
fig.savefig(PLOT_OUT, dpi=150, bbox_inches="tight")
print("\nGrafik disimpan pada:", PLOT_OUT)
plt.show()


## 9. Pemeriksaan ukuran bobot dan konversi fp16

GitHub menolak berkas di atas 100 MB, dan berkas di atas 50 MB memunculkan peringatan. Sel ini
memeriksa ukuran bobot hasil pelatihan; bila melewati 100 MB, bobot disimpan ulang dalam **fp16**
lalu `evaluate.py` dijalankan ulang untuk membuktikan skornya tidak berubah.

Pada percobaan ConvNeXt-Tiny, bobot fp32 berukuran 106,2 MB sehingga konversi diperlukan dan
menghasilkan 53,1 MB. Pada EfficientNetV2-S, bobot fp32 berukuran 77,8 MB sehingga tidak perlu
konversi.

Dasar keamanan konversi: seluruh bobot ConvNeXt berupa tensor float tanpa `num_batches_tracked`,
dan `load_state_dict` mengubahnya kembali ke fp32 saat dimuat. Pengujian pada 413 gambar beku
menghasilkan macro-F1 identik (0,9658 pada fp32 maupun fp16) dengan 0 prediksi berbeda; error
relatif setiap bobot terukur 4,880e-04, tepat batas presisi fp16 (2^-11).


In [ ]:
mb = OUT_WEIGHTS.stat().st_size / 2 ** 20
print("Bobot hasil pelatihan : %.1f MB (%s byte)" % (mb, format(OUT_WEIGHTS.stat().st_size, ",")))
print("Batas keras GitHub    : 100 MB")

if mb > 100:
    import torch

    print("\nUkuran melewati batas; bobot disimpan ulang sebagai fp16.")
    shutil.copyfile(OUT_WEIGHTS, WORK_DIR / "model_fp32.pth")
    sd = torch.load(OUT_WEIGHTS, map_location="cpu", weights_only=True)
    non_float = [k for k, v in sd.items() if not v.is_floating_point()]
    if non_float:
        raise RuntimeError("terdapat tensor non-float sehingga fp16 tidak aman: %s" % non_float)

    torch.save({k: v.half() for k, v in sd.items()}, OUT_WEIGHTS)
    print("Ukuran setelah fp16   : %.1f MB" % (OUT_WEIGHTS.stat().st_size / 2 ** 20))

    run([sys.executable, REPO_DIR / "evaluate.py",
         "--submission", WORK_DIR, "--data", "my_val", "--out", EVAL_FP16])

    m32 = json.loads(EVAL_OUT.read_text(encoding="utf-8"))
    m16 = json.loads(EVAL_FP16.read_text(encoding="utf-8"))
    print("\nmacro-F1 fp32 %.4f  ->  fp16 %.4f  (selisih %+.4f)"
          % (m32["macro_f1"], m16["macro_f1"], m16["macro_f1"] - m32["macro_f1"]))
    print("Confusion matrix identik:", m32["confusion_matrix"] == m16["confusion_matrix"])
    if abs(m16["macro_f1"] - m32["macro_f1"]) > 0.005:
        print("[PERINGATAN] selisih melebihi 0,005; versi fp16 ini jangan di-commit.")
else:
    print("\nUkuran aman untuk GitHub; konversi tidak diperlukan.")

print("sha256:", hashlib.sha256(OUT_WEIGHTS.read_bytes()).hexdigest()[:16], "...")


## 10. Promosi ke `submission/`

`submission/` selalu diisi model **terbaik** di antara kedua percobaan yang dibandingkan, yaitu
ConvNeXt-Tiny crop 224 (0,9658) dan EfficientNetV2-S crop 288 (0,9595). Aturannya sederhana:
bila skor percobaan ini melebihi skor tertinggi dari keduanya, bobot dipromosikan; bila tidak,
`submission/` dibiarkan seperti sekarang. Dengan begitu urutan menjalankan notebook tidak
berpengaruh — siapa pun yang menang akan berakhir di `submission/`.

Promosi selalu menyalin dua berkas sekaligus, yaitu `model.py` dan `model.pth`. Bila hanya salah
satunya ter-commit, `strict=True` akan gagal dan submission tidak dapat dinilai sama sekali.

**Perlu diingat meski tidak menghalangi promosi:** split lokal hanya berisi 413 gambar sehingga
satu gambar bernilai sekitar 0,0024 poin macro-F1. Kemenangan dengan selisih di bawah 0,01 belum
dapat dipercaya sebagai perbaikan sungguhan; sel ini akan mencetak peringatan bila hal itu terjadi.


In [ ]:
skor = max(json.loads(HIST_OUT.read_text(encoding="utf-8")), key=lambda h: h["val_macro_f1"])["val_macro_f1"]
terbaik_dua = max(SKOR_TERCATAT.values())

print("Skor percobaan ini            : %.4f  (%s)" % (skor, PERCOBAAN_INI))
print("Skor tertinggi dari keduanya  : %.4f  (%s)"
      % (terbaik_dua, max(SKOR_TERCATAT, key=SKOR_TERCATAT.get)))
print("Selisih                       : %+.4f" % (skor - terbaik_dua))
print()

dipromosikan = skor > terbaik_dua

if dipromosikan:
    if skor - terbaik_dua < 0.01:
        print("[PERINGATAN] selisih di bawah 0,01 (satu gambar pada split ini bernilai 0,0024).")
        print("            Kemenangannya belum dapat dipercaya sepenuhnya, tetapi tetap")
        print("            dipromosikan sesuai aturan pemilihan otomatis.")
    shutil.copyfile(REPO_DIR / CANDIDATE, REPO_DIR / "submission" / "model.py")
    shutil.copyfile(OUT_WEIGHTS, REPO_DIR / "submission" / "model.pth")
    print("Dipromosikan ke submission/: model.py dan model.pth diperbarui.")
else:
    print("Tidak dipromosikan: skor percobaan ini tidak melebihi model terbaik yang ada.")
    print("submission/ tetap memakai model dengan skor %.4f." % terbaik_dua)

print()
print("Isi submission/ saat ini:")
for path, label in [(REPO_DIR / "submission" / "model.py", "model.py"),
                    (REPO_DIR / "submission" / "model.pth", "model.pth"),
                    (REPO_DIR / "submission" / "requirements.txt", "requirements.txt")]:
    if path.is_file() and path.stat().st_size > 0:
        ukuran = ("%.1f MB" % (path.stat().st_size / 2 ** 20)) if path.suffix == ".pth" else ("%d B" % path.stat().st_size)
        print("  [OK]     %-18s %s" % (label, ukuran))
    else:
        print("  [HILANG] %-18s" % label)

pth = REPO_DIR / "submission" / "model.pth"
if pth.is_file():
    print()
    print("  ukuran  : %.1f MB (batas 100 MB GitHub; batas ketentuan 500 MB)" % (pth.stat().st_size / 2 ** 20))
    print("  sha256  : %s..." % hashlib.sha256(pth.read_bytes()).hexdigest()[:16])


## 11. Pengunduhan berkas dan perintah commit

Berkas hasil disalin ke Drive sebagai arsip dan diunduh ke laptop. Commit dilakukan dari laptop,
bukan dari Colab, agar riwayat commit tetap terbaca dan repositori tetap publik.


In [ ]:
from google.colab import files

for f in [OUT_WEIGHTS, PLOT_OUT, HIST_OUT, EVAL_OUT]:
    if pathlib.Path(f).is_file():
        shutil.copyfile(f, CKPT_DIR / pathlib.Path(f).name)
print("Disalin ke Drive:", CKPT_DIR)

for f in [OUT_WEIGHTS, PLOT_OUT, HIST_OUT]:
    if pathlib.Path(f).is_file():
        files.download(str(f))

print()
print("CATATAN: promosi pada langkah 10 terjadi di dalam repositori hasil clone di Colab.")
print("Repositori di laptop tidak berubah dengan sendirinya; samakan dengan perintah berikut.")
print()
if dipromosikan:
    print("Percobaan ini menang, sehingga submission/ perlu diperbarui.")
else:
    print("Percobaan ini tidak menang; submission/ tidak perlu diubah, cukup diarsipkan.")
print()
print("Perintah pada laptop (repositori sudah di-clone), ganti <unduhan> dengan folder:")
print("  cp <unduhan>/%s.pth        runs/%s/model.pth" % (EXP_NAME, EXP_NAME))
print("  cp <unduhan>/history.json  runs/%s/history.json" % EXP_NAME)
if dipromosikan:
    print("  cp %s submission/model.py" % CANDIDATE)
    print("  cp <unduhan>/%s.pth        submission/model.pth" % EXP_NAME)
print("  python evaluate.py --submission ./submission --data ./my_val")
print("  git add runs/ submission/")
print('  git commit -m "komparasi convnext_tiny vs efficient_v2s: %s macro-F1 %.4f"' % (EXP_NAME, skor))
print("  git push")


## 12. Penanganan kendala

| Gejala | Penyebab dan tindakan |
|---|---|
| `CUDA: False` | GPU belum dialokasikan. Jalankan `Runtime` → `Change runtime type` → `T4 GPU`, lalu ulangi sel. Bila GPU sedang terpakai penuh, tunggu beberapa menit. |
| Clone meminta nama pengguna dan kata sandi | Repositori tidak publik atau URL salah. Repositori wajib publik sejak pendaftaran sampai nilai diumumkan. |
| `berkas zip dataset tidak ditemukan` | Jalur pada Drive berbeda. Periksa `MyDrive/Studio 2/Dataset/train.zip`; nama folder bersifat peka huruf besar dan kecil. |
| Sesi Colab terputus di tengah pelatihan | Colab memutus sesi yang menganggur sekitar 90 menit. `train.py` selalu menyimpan checkpoint terbaik, dan berkas hasil disalin ke Drive setiap kali notebook selesai dijalankan. |
| `[GAGAL] model.py tidak mendefinisikan get_model()` | Berkas arsitektur yang ditunjuk salah atau tidak ada pada repositori. Periksa isi folder `experiments/`. |
| `Error(s) in loading state_dict ... strict=True` | `model.py` dan `model.pth` tidak sepasang. Jangan meng-commit hanya salah satunya. |
| Kehabisan memori GPU saat melatih | Turunkan `BATCH_SIZE`. Pada crop 288, batch 32 perlu VRAM lebih besar daripada crop 224. |
| Bobot ditolak saat push karena melebihi 100 MB | Langkah 9 belum dijalankan atau gagal; bobot harus tersimpan dalam fp16. |
| Skor hasil pelatihan ulang berbeda dari skor tercatat | Hal yang wajar. Selisih beberapa per seribu berasal dari urutan augmentasi acak; aman selama masih di bawah 0,01. |
| Skor preview berbeda jauh dari skor lokal | Split lokal (413 gambar) berbeda dari validation set tersembunyi (±740 gambar). Split lokal dipakai untuk memilih model, skor preview untuk memeriksa generalisasi. |


## 13. Tabel hasil antar-percobaan

Seluruh angka diukur pada split validasi beku yang sama (413 gambar, seed 42) dengan `evaluate.py`
resmi. Satu gambar bernilai ±0,0024 poin, sehingga selisih di bawah ±0,01 belum dapat dipercaya
sebagai perbaikan.

| Percobaan | Backbone | Crop | Augmentasi | lr / wd | batch | Epoch | Epoch terbaik | Macro-F1 | Akurasi | Salah | Catatan |
|---|---|---|---|---|---|---|---|---|---|---|---|
| eksperimen 1 | EfficientNetV2-S | 288 | standar | 3e-4 / 1e-4 | — | 12 | 3 | 0,9555 | 0,9564 | 18 | val loss melonjak mulai epoch 6 |
| eksperimen 2 | EfficientNetV2-S | 288 | strong | 1,5e-4 / 1e-2 | 32 | 12 | 9 | 0,9590 | 0,9613 | 16 | dijalankan di Colab |
| eksperimen 3 | EfficientNetV2-S | 288 | strong | 1,5e-4 / 1e-2 | 32 | 11 | 7 | 0,9595 | 0,9613 | 16 | lokal |
| eksperimen 4 | ConvNeXt-Tiny | 224 | strong | 1e-4 / 1e-2 | 16 | 20 → 9 | 5 | 0,9658 | 0,9685 | 13 | submission saat ini |

Angka pada kolom `batch` untuk eksperimen 1 tidak terekam karena `train.py` belum menuliskan
argumennya ke berkas riwayat pada saat itu. Nilai untuk eksperimen 3 mengikuti eksperimen 2,
sesuai catatan eksperimen yang menyatakan kedua konfigurasi tersebut sama.
